In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Parameters

In [4]:
name_dataset = 'SSTv2'
name_model = 'gpt-4o'
mode = 'zero'
seed = 1
part = 2

In [5]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post9/df_test_{part}.csv'

In [6]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post10/df_test_{part}.csv'

In [7]:
path_credentials = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/credentials/OPENAI_API_KEY.json'

# 2. Load Environment

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import json
import requests
import pandas as pd
from openai import OpenAI

In [10]:
with open(path_credentials, "r") as f:
    credentials = json.load(f)

In [11]:
OPENAI_API_KEY = credentials["OPENAI_API_KEY"]

In [12]:
client = OpenAI(api_key=OPENAI_API_KEY)

# 3. Functions

In [13]:
def zero_shot_prompt(text):

    intro = (
        "Classify the sentiment of the following movie sentence from the Stanford Sentiment Treebank v2 (SST-2) dataset.\n"
        "Respond only with a single digit: 0 if the sentiment is negative, "
        "1 if the sentiment is positive.\n"
    )

    target = f"Sentence: \"{text}\"\nLabel:"

    return intro + target

In [14]:
def predict_label(text, prompt):

    try:

        start_time = time.perf_counter()
        first_token_time = None
        output_text = ""

        stream = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            top_p=1.0,
            seed=seed,
            stream=True,
            stream_options={"include_usage": True}
        )

        for chunk in stream:
            if first_token_time is None and len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                first_token_time = time.perf_counter()

            if len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                output_text += chunk.choices[0].delta.content

            if chunk.usage is not None:
                usage = chunk.usage

        end_time = time.perf_counter()

        return {
            "prediction": output_text.strip(),
            "latency_ms": (end_time - start_time) * 1000,
            "ttft_ms": (
                (first_token_time - start_time) * 1000
                if first_token_time else None
            ),
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens
        }
    
    except:
        
        return {
            "prediction": '-1',
            "latency_ms": '-',
            "ttft_ms": '-',
            "input_tokens": '-',
            "output_tokens": '-'
        }

# 4. Load Dataset

In [15]:
df = pd.read_csv(path_open)

In [16]:
df.shape

(2021, 21)

In [17]:
pred_label = []
pred_latency = []
pred_ttft = []
pred_input_tokens = []
pred_output_tokens = []

In [18]:
for i in range(len(df)):

  text = df['text'].iloc[i]
  prompt = zero_shot_prompt(text)
  output = predict_label(text, prompt)

  try:

    pred_label.append(int(output['prediction']))
    pred_latency.append(output['latency_ms'])
    pred_ttft.append(output['ttft_ms'])
    pred_input_tokens.append(output['input_tokens'])
    pred_output_tokens.append(output['output_tokens'])
  
  except:

    pred_label.append(-1)
    pred_latency.append('-')
    pred_ttft.append('-')
    pred_input_tokens.append('-')
    pred_output_tokens.append('-')

  if (i % 10) == 0:
    print(i)

0


10


20


30


40


...

1980


1990


2000


2010


2020


In [19]:
df[f'{name_model}-{mode}-seed-{seed}-label'] = pred_label
df[f'{name_model}-{mode}-seed-{seed}-latency'] = pred_latency
df[f'{name_model}-{mode}-seed-{seed}-ttft'] = pred_ttft
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'] = pred_input_tokens
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'] = pred_output_tokens

In [20]:
df[f'{name_model}-{mode}-seed-{seed}-label'].value_counts()

,count
gpt-4o-zero-seed-1-label,
0,1149
1,872


In [21]:
df[f'{name_model}-{mode}-seed-{seed}-latency'].describe()

,gpt-4o-zero-seed-1-latency
count,2021.000000
mean,400.780099
std,141.416209
min,229.474608
25%,343.771630
50%,380.894769
75%,418.222890
max,2038.310048


In [22]:
df[f'{name_model}-{mode}-seed-{seed}-ttft'].describe()

,gpt-4o-zero-seed-1-ttft
count,2021.000000
mean,390.691746
std,141.425115
min,218.033569
25%,333.644957
50%,370.208759
75%,407.604783
max,2025.998288


In [23]:
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'].describe()

,gpt-4o-zero-seed-1-input-tokens
count,2021.000000
mean,72.765957
std,8.711444
min,63.000000
25%,66.000000
50%,70.000000
75%,77.000000
max,115.000000


In [24]:
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'].describe()

,gpt-4o-zero-seed-1-output-tokens
count,2021.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


# 5. Save Dataset

In [25]:
df.to_csv(path_save, index = False)

# 6. Execution Time

In [26]:
end_notebook = time.time()

In [27]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 15m 34.76s
